# GPT-2 Fine-Tuning with Low-Rank Adaptation (LoRA)

**Task:** Generate descriptive tags for English-language quotations using parameter-efficient fine-tuning.

**Approach:** Compare pretrained GPT-2 against a LoRA-adapted version using the same prompts and decoding settings. This notebook records an experimental workflow, not a production-ready tagging system.

## 1. Environment setup

Install the dependencies and initialize PyTorch, Hugging Face Transformers, Datasets, and PEFT. A CUDA GPU is recommended for training. The recorded run used a T4 GPU.

In [1]:
%pip install -q "huggingface-hub==0.36.2" "transformers==4.57.6" "peft==0.18.0" "datasets>=3,<5" "accelerate>=0.33,<2"

In [2]:
import os
import random

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Select a T4 GPU in Colab before training.")

Device: cuda
GPU: Tesla T4


## 2. Load GPT-2 and its tokenizer

Load the pretrained GPT-2 causal language model and tokenizer. GPT-2 predicts the next token given preceding text; LoRA adapts the model to the quotation-tagging task.

In [3]:
MODEL_ID = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
base_model.config.pad_token_id = tokenizer.pad_token_id

parameter_count = sum(param.numel() for param in base_model.parameters())
print(f"Total base parameters: {parameter_count:,}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total base parameters: 124,439,808


## 3. Prepare quotation-tagging data

Load `Abirate/english_quotes` from Hugging Face, split it into training and held-out subsets, and format each example as `quotation ->: tags`. The delimiter converts structured tags into a text-completion target. **Limitation:** This training format predicts tokens for the entire sequence, not exclusively the tag span.

In [4]:
raw = load_dataset("Abirate/english_quotes", split="train")
print(f"Columns: {raw.column_names}; rows: {len(raw)}")
print(raw[0])

splits = raw.train_test_split(test_size=0.1, seed=SEED)

def format_example(row):
    return {"prediction": f"{row['quote']} ->: {row['tags']}"}

train_text = splits["train"].map(format_example)
test_text = splits["test"].map(format_example)

print(f"Train: {len(train_text)}; test: {len(test_text)}")
print(train_text[0]["prediction"][:500])

Columns: ['quote', 'author', 'tags']; rows: 2508
{'quote': '“Be yourself; everyone else is already taken.”', 'author': 'Oscar Wilde', 'tags': ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']}


Map:   0%|          | 0/2257 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Train: 2257; test: 251
“If I got rid of my demons, Iâ€™d lose my angels.” ->: ['angels', 'demons']


## 4. Establish a pretrained-model baseline

Generate tags for a small set of quotations before fine-tuning. These baseline outputs provide a qualitative reference; they are not a formal accuracy measurement.

In [5]:
examples = [
    "Life is like a box of chocolates, you never know what you are gonna get",
    str(test_text[0]["quote"]),
    str(test_text[1]["quote"]),
]

def predict_tags(model, quote, max_new_tokens=30):
    prompt = f"{quote} ->: "
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=900
    ).to(device)
    model.eval()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion_ids = generated_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

baseline = [predict_tags(base_model, quote) for quote in examples]
for quote, prediction in zip(examples, baseline):
    print(f"QUOTE: {quote[:150]}")
    print(f"BASELINE TAGS: {prediction}")
    print("-" * 70)

QUOTE: Life is like a box of chocolates, you never know what you are gonna get
BASELINE TAGS: ~~~

I'm not sure if I'm going to be able to get a good night's sleep, but I'm sure I'll be
----------------------------------------------------------------------
QUOTE: “I don't mind making jokes, but I don't want to look like one.”
BASELINE TAGS: ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ
----------------------------------------------------------------------
QUOTE: “Books are not made to be believed, but to be subjected to inquiry. When we consider a book, we mustn't ask ourselves what it says but what it means..
BASELINE TAGS: 《The Book of Mormon““““““““““““
----------------------------------------------------------------------


## 5. Configure LoRA adapters

Freeze the pretrained model parameters and insert LoRA adapters into GPT-2 attention projections. This reduces the number of trainable parameters compared with updating the full model.

In [6]:
for param in base_model.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn"],
    fan_in_fan_out=True,  # GPT-2 uses Conv1D projections.
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 589,824 || all params: 125,029,632 || trainable%: 0.4717


## 6. Tokenize and train

Tokenize examples to a maximum sequence length of 256 and fine-tune for 500 optimizer steps. The training loss is reported below. **Caution:** No validation loss is computed in this run, and the training loss alone does not establish tagging quality.

In [7]:
MAX_LENGTH = 256

def tokenize(batch):
    return tokenizer(batch["prediction"], truncation=True, max_length=MAX_LENGTH)

train_tokens = train_text.map(
    tokenize, batched=True, remove_columns=train_text.column_names
)
test_tokens = test_text.map(
    tokenize, batched=True, remove_columns=test_text.column_names
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="outputs/training",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    max_steps=500,
    learning_rate=2e-4,
    logging_steps=25,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    fp16=(device == "cuda"),
    dataloader_num_workers=0,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokens,
    data_collator=collator,
)
train_result = trainer.train()
print(f"Final training loss: {train_result.training_loss:.4f}")

Map:   0%|          | 0/2257 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
25,4.176300
50,4.133600
75,3.701800
100,3.272100
125,2.966800
150,2.982900
175,2.967800
200,2.862800
225,2.866400
250,2.851300


Final training loss: 3.0486


## 7. Save the trained adapter

Save the PEFT adapter and tokenizer separately from the full pretrained GPT-2 weights. The output directory is generated locally and is excluded from the Git repository.

In [8]:
ADAPTER_DIR = "outputs/gpt2_quote_lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter files:", os.listdir(ADAPTER_DIR))

Saved adapter files: ['vocab.json', 'tokenizer.json', 'merges.txt', 'tokenizer_config.json', 'README.md', 'special_tokens_map.json', 'adapter_model.safetensors', 'adapter_config.json']


## 8. Compare generations after fine-tuning

Use the same quotations and deterministic decoding as the baseline to compare behavior before and after training.

In [9]:
model.config.use_cache = True
adapted = [predict_tags(model, quote) for quote in examples]

for quote, before, after in zip(examples, baseline, adapted):
    print(f"QUOTE: {quote[:180]}")
    print(f"BEFORE: {before}")
    print(f"AFTER:  {after}")
    print("-" * 70)

QUOTE: Life is like a box of chocolates, you never know what you are gonna get
BEFORE: ~~~

I'm not sure if I'm going to be able to get a good night's sleep, but I'm sure I'll be
AFTER:  ~~~The Good Life, ~~~The Good Life, ~~~The Good Life, ~~~The Good Life, ~~~
----------------------------------------------------------------------
QUOTE: “I don't mind making jokes, but I don't want to look like one.”
BEFORE: ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ
AFTER:  vernacular', 'humor', 'humor-inspirational', 'humor-inspirational-comedies', 'humor-
----------------------------------------------------------------------
QUOTE: “Books are not made to be believed, but to be subjected to inquiry. When we consider a book, we mustn't ask ourselves what it says but what it means...”
BEFORE: 《The Book of Mormon““““““““““““
AFTER:  vernacular', 'books', 'books-in-words', 'books', 'books-in-words-in-words', 'books-
----------------------------------------------------------------------


## 9. Inspect held-out examples and limitations

Compare generated tags with reference tags for five held-out quotations. This qualitative inspection does not measure precision, recall, or exact-match accuracy. The observed generations may be repetitive or inaccurate. Future work should add quantitative evaluation, validation-based checkpoint selection, and tag-only loss masking.

In [10]:
for index, row in enumerate(test_text.select(range(min(5, len(test_text)))), start=1):
    print(f"EXAMPLE {index}")
    print(f"Quote: {str(row['quote'])[:220]}")
    print(f"Reference tags: {row['tags']}")
    print(f"Generated tags: {predict_tags(model, str(row['quote']))}")
    print("-" * 70)

EXAMPLE 1
Quote: “I don't mind making jokes, but I don't want to look like one.”
Reference tags: ['appearance', 'jokes', 'marilyn-monroe']
Generated tags: vernacular', 'humor', 'humor-inspirational', 'humor-inspirational-comedies', 'humor-
----------------------------------------------------------------------
EXAMPLE 2
Quote: “Books are not made to be believed, but to be subjected to inquiry. When we consider a book, we mustn't ask ourselves what it says but what it means...”
Reference tags: ['books']
Generated tags: vernacular', 'books', 'books-in-words', 'books', 'books-in-words-in-words', 'books-
----------------------------------------------------------------------
EXAMPLE 3
Quote: “The loneliest moment in someoneâ€™s life is when they are watching their whole world fall apart, and all they can do is stare blankly.”
Reference tags: ['loneliness']
Generated tags: vernacular', 'life', 'life-in-the-world', 'life-in-the-world-of-the-lonel
-----------------------------------------------